# Lab: Separate More Complex Data
## Purpose:
- Understand the effect of adding a hidden layer to a neural network.
- Understand the role of a non-linear activation function.

### Topics:
- Hidden layers
- Non-linear activation functions (SoftMax)

### Tasks
Imagine that you want a model to choose the correct next token from:
- "mat"
- "apple"
- "bank"

But remember, the word "bank" has two meanings:
- a river bank
- a financial institution

Therefore, the embeddings for "mat" will form a cluster, and the embeddings for "apple" will cluster, but the embeddings for "bank" will exist in 2 separate clusters.

**In this lab, you will**:
* Visualize and compare the decision boundaries of:
    * a single-layer neural network with multiple neurons
    * a multi-layer neural network without an activation function
    * a multi-layer neural netework with a ReLU activation function

Date: 2026-02-21

Source: https://colab.research.google.com/github/google-deepmind/ai-foundations/blob/master/course_3/gdm_lab_3_3_separate_more_complex_data.ipynb

References: https://github.com/google-deepmind/ai-foundations
- GDM GH repo used in AI training courses at the university & college level.

In [ ]:
%%capture
# Install the custom package for this course.
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

import os # For adjusting Keras settings.
os.environ["KERAS_BACKEND"] = "jax" # Set a parameter for Keras.

# Packages used.
import jax.numpy as jnp # For converting the embeddings and labels to matrices.
import pandas as pd # For loading the dataset.
from ai_foundations import machine_learning # For defining and training MLPs.
from ai_foundations import visualizations # For visualizing data and boundaries.

## Load and plot the data

To get started, run the following cell to load the dataset that contains the embeddings and labels. Note that by convention, the matrix that contains the input data, in this case the embeddings, is usually called `X`. The vector that contains the target labels is called `y`.

This convention will be used from now on.

In [ ]:
# Load data using pandas.
df = pd.read_csv("https://storage.googleapis.com/dm-educational/assets/ai_foundations/mat-apple-bank-dataset.csv")

# Extract embeddings (Embedding_dim_1, Embedding_dim_2) and labels.
X = jnp.array(df[["Embedding_dim_1", "Embedding_dim_2"]].values)
# Labels: 0 ("mat"), 1 ("apple"), or 2 ("bank").
y = jnp.array(df["Label"].values)

# Human-readable labels.
labels = ["mat", "apple", "bank"]

In [ ]:
# Plot the points
visualizations.plot_data_and_mlp(X, y, labels, title="2D Embeddings of Sentences")

## Train a single-layer model

This model has a layer with **multiple neurons** and uses a **SoftMax** activation function. This introduces multiple lines as decision boundaries, instead of just one.

Each time this cell is run, it will train the single-layer model and generate the visualization of the decision boundary.

------
> **ℹ️ Info: How to interpret decision boundary visualizations**
>
> In a decision boundary visualization,shaded areas are shown in multiple colors. Each of the colors represents one class as predicted by the model. For example, the area that is shaded in green represents the set of all points for which the model predicts the class "bank" because it assigns the highest probability to that class. If a data point falls within the area of its color, then the model makes a correct prediction (e.g., all green points within the area shaded in green are correct predictions). If a data point falls within an area of a different color, then the true and the predicted class are different and the model misclassified the data point.
------
The model divides the space into three areas separated by two straight lines. Regardless of how many times the training is run, it will never perfectly separate the three classes because a single-layer model can only learn linear decision boundaries.

Above the plot, the code also prints the **accuracy** of the model on the training data. The accuracy is a metric that captures the percentage of examples that have been correctly classified. An accuracy of 100% means that every example in the dataset has been assigned the correct class by the model.

In [ ]:
# The parameter hidden_dims defines the dimensions of each hidden layer.
# If an empty list is passed, as is done here, the MLP will be initialized
# without any hidden layers, resulting in a single-layer neural network.
#
# Note that you will learn in subsequent labs how to implement an MLP in Python.
# For now, you can use `build_mlp` for defining an MLP, and `train_mlp` for
# training it.
model_1layer = machine_learning.build_mlp(hidden_dims=[], n_classes=3)

history = machine_learning.train_mlp(model_1layer, X, y, 200)

print(f"Accuracy (single layer model): {history.history['accuracy'][-1]*100:.2f}%")

visualizations.plot_data_and_mlp(
    X,
    y,
    labels,
    mlp_model=model_1layer,
    title="Decision Boundaries - Single Layer Model",
)

## Train a multi-layer neural network

Multi-layer models are capable of more nuance. The next cell trains a model with an additional **hidden layer**.

Run the following cell to train a multi-layer neural network and visualize its new decision boundary.

As a first step, visualize the following model. It implements multiple layers but *does not* use an **activation function** in its **hidden layer**.

Without the non-linear activation function, the model is still stick with straight lines.

In [ ]:
# Linear activation function: activation="linear"
hidden_dims = [10]          # number of neurons in the layer.
model_2layer = machine_learning.build_mlp(
    hidden_dims=hidden_dims, n_classes=3, activation="linear"
)

history = machine_learning.train_mlp(model_2layer, X, y, 200)

print(f"Accuracy (multi-layer model): {history.history['accuracy'][-1]*100:.2f}%")
visualizations.plot_data_and_mlp(
    X,
    y,
    labels,
    mlp_model=model_2layer,
    title="Decision Boundaries - Multi-Layer Model",
)

### Add the activation function

The following network is exactly the same as the one above but uses a **ReLU** activation function to compute the output of the hidden layer.
The non-linear activation function can produce curves and arcs instead of just straight lines.

In [ ]:
# Linear activation function: activation="relu"
# hidden_dims = [10]
model_2layer = machine_learning.build_mlp(
    hidden_dims=hidden_dims, n_classes=3, activation="relu"
)

history = machine_learning.train_mlp(model_2layer, X, y, 200)

print(f"Accuracy (multi-layer model): {history.history['accuracy'][-1]*100:.2f}%")
visualizations.plot_data_and_mlp(
    X,
    y,
    labels,
    mlp_model=model_2layer,
    title="Decision Boundaries - Multi-Layer Model",
)

## Mathmatical Proof

Given a two-layer neural network *without* an **activation function** in the **hidden layer** that has:
* an input layer with two features
* a hidden layer with one neuron
* an output layer with one neuron.

Assume that the bias terms are all 0 so they can be ignored.

Then the weighted sum in the hidden layer with weights $w^{(h)}$ is:

$$z^{(h)}_1=w_1^{(h)}x_1 + w_2^{(h)}x_2$$

<br />

The output of the hidden layer, $y^{(h)}$ is then the same as $z^{(h)}$ since there is no non-linear activation function:

$$y^{(h)}_1 = z^{(h)}_1$$

<br />

Now consider the output layer. The input to the output layer $x^{(o)}$ is the output of the hidden layer $y^{(h)}$. The weighted sum $z^{(o)}$ in the hidden layer is therefore:

$$z^{(o)} = w^{(o)}_1 y^{(h)}_1$$

<br />

You can now substitute $y^{(h)}_1$ by $z^{(h)}_1$, since $y^{(h)}_1 = z^{(h)}_1$ and then substitute and simplify further using the equations described:

\begin{align}
z^{(o)} &= w^{(o)}_1 y^{(h)}_1 \\ &= w^{(o)}_1 z^{(h)}_1 \\ &= w^{(o)}_1 \left(w_1^{(h)}x_1 + w_2^{(h)}x_2\right) \\ &= w^{(o)}_1w_1^{(h)}x_1 + w^{(o)}_1w_2^{(h)}x_2
\end{align}

<br />

Now if you define new weights $w_1'$ and $w_2'$ such that $w_1' = w^{(o)}_1w_1^{(h)}$ and $w_2' = w^{(o)}_1w_2^{(h)}$, you can define the output layer as:

$$z^{(o)} = w_1'x_1 + w_2'x_2$$

<br />

Note that this is now the equation of **a single-layer neural network with two inputs**. This is no coincidence,

### any multi-layer neural network without a non-linear activation function can also be expressed as a single-layer neural network.
Therefore a network without the activation function cannot learn more complex functions that a single-layer MLP.

However, if there is a non-linear activation function applied to the weighted sum in the hidden layer, then this reduction of a multi-layer MLP to a single layer MLP is no longer possible and the model can learn more complex decision boundaries.